# 01 — Triplet Parsing (CholecT50)

**Goal:** Load the CholecT50 JSON annotations, understand the binary-to-label mapping,
and parse the surgical action triplets `(instrument, verb, target)` for 3 sample videos.

**Videos:** VID01 (short), VID05 (medium), VID40 (longer)

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from pathlib import Path
from src.triplet_parser import (
    load_categories,
    print_categories,
    parse_video,
    parse_multiple_videos,
    video_summary,
)

# Project paths
PROJECT_ROOT = Path('..').resolve()
TRIPLET_DIR = PROJECT_ROOT / 'data' / 'CholecT45' / 'triplet'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'parsed_triplets'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Triplet dir:', TRIPLET_DIR)
print('Available:', os.listdir(TRIPLET_DIR))

## 1. Inspect Category Dictionaries

Categories are embedded in each video JSON. We load them once and inspect.

In [ ]:
categories = load_categories(TRIPLET_DIR / 'VID01.json')
print_categories(categories)

## 2. Parse a Single Video (VID01)

Using `parse_video()` from `src/triplet_parser.py`, which:
- Loads the JSON
- Decodes each 15-element annotation entry into `(instrument, verb, target)`
- Expands multi-label frames (each active triplet → one row)
- Returns a tidy DataFrame

In [ ]:
df_vid01 = parse_video(TRIPLET_DIR / 'VID01.json')
print(f'VID01: {len(df_vid01)} triplet rows across {df_vid01["frame"].nunique()} frames')
df_vid01.head(10)

## 3. Multi-Label Frame Analysis

How many frames have multiple simultaneous triplets?

In [ ]:
triplets_per_frame = df_vid01.groupby('frame').size()

print('=== Triplets per Frame Distribution (VID01) ===')
print(triplets_per_frame.value_counts().sort_index())
print(f'\nMax triplets in a single frame: {triplets_per_frame.max()}')
print(f'Mean triplets per frame: {triplets_per_frame.mean():.2f}')
print(f'Frames with >1 triplet: {(triplets_per_frame > 1).sum()} / {len(triplets_per_frame)}')

In [ ]:
# Show a multi-label frame example
multi_frames = df_vid01[df_vid01['num_triplets_in_frame'] > 1]
if not multi_frames.empty:
    sample_frame = multi_frames['frame'].iloc[0]
    print(f'=== Multi-label Frame {sample_frame} ===')
    display(df_vid01[df_vid01['frame'] == sample_frame][['frame', 'instrument', 'verb', 'target', 'triplet_label']])

## 4. Label Frequency Summary

In [ ]:
print('=== VID01 Label Frequencies ===')
print('\nTop 10 Triplets:')
print(df_vid01['triplet_label'].value_counts().head(10))
print('\nInstruments:')
print(df_vid01['instrument'].value_counts())
print('\nVerbs:')
print(df_vid01['verb'].value_counts())
print('\nTargets:')
print(df_vid01['target'].value_counts())

---
**Next step:** Parse all 3 sample videos (VID01, VID05, VID40), save CSVs, and build the graph.